In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
filename = "replan_@MAEDeR_maze-128-128-1-even-1-k50_2026-03-20-17:31_seed123_25delays"
filepath = os.path.join(os.path.dirname(os.path.abspath("__file__")), "output", "maze1", f"{filename}.json")
complete_result = {
    "@MAEDeR": json.load(open(filepath, "r")),
    "FlexSIPP": json.load(open(filepath.replace("@MAEDeR", "FlexSIPP"), "r"))
}

In [ ]:
df = pd.DataFrame(columns=["Delay Idx", "Delay Agent", "Delay Amount", "Delayed Starttime", "Total FlexSIPP" ,"Total @MAEDeR"])
rows = 0
paths = {
    "FlexSIPP": {a: [r["arrival"][1]]  for a, r in complete_result["FlexSIPP"]["delay0"]["initial_paths"].items()},
    "@MAEDeR": {a: [r["arrival"][1]]  for a, r in complete_result["@MAEDeR"]["delay0"]["initial_paths"].items()}
}
not_solved = {"FlexSIPP": 0, "@MAEDeR": 0}
for alg, result in complete_result.items():
    delays = ["" for a in result["delay0"]["initial_paths"]]
    for delay in result:
        if result[delay] and result[delay]["unique_routes_safe"]:
            for a, r in result[delay]["arrival_times"].items():
                paths[alg][a].append(r["arrival"][1])
            delays[int(result[delay]["delay_agent"])-1] = str(result[delay]["delay_agent"])
            print("Delay", alg, result[delay]["delay_agent"], paths[alg][str(result[delay]["delay_agent"])])
            assert complete_result["FlexSIPP"][delay]["delay_agent"] == complete_result["@MAEDeR"][delay]["delay_agent"]
            if alg == "FlexSIPP":
                df.loc[rows] = [
                    delay,
                    complete_result["FlexSIPP"][delay]["delay_agent"],
                    complete_result["FlexSIPP"][delay]["delayed_start_time"],
                    complete_result["FlexSIPP"][delay]["delayed_start_time"] - complete_result["FlexSIPP"][delay]["original_start_time"],
                    sum([paths["FlexSIPP"][a][-1] - paths["FlexSIPP"][a][0] for a in paths["FlexSIPP"]]),
                    sum([paths["@MAEDeR"][a][-1] - paths["@MAEDeR"][a][0] for a in paths["@MAEDeR"]])
                ]
                rows += 1
        else:
            not_solved[alg] += 1

In [ ]:
print(not_solved)
print("\t\t@MAEDEr FLexSIPP")
for a in paths["FlexSIPP"]:
    print("Agent", a, "\t", paths["@MAEDeR"][a][-1] - paths["@MAEDeR"][a][0],"\t", paths["FlexSIPP"][a][-1] - paths["FlexSIPP"][a][0])
print("Agent", a, "\t", 
    sum([(paths["@MAEDeR"][a][-1] - paths["@MAEDeR"][a][0]) for a in paths["FlexSIPP"]]),"\t", 
    sum([(paths["FlexSIPP"][a][-1] - paths["FlexSIPP"][a][0]) for a in paths["@MAEDeR"]])
)


In [ ]:
df

In [ ]:
fig, ax = plt.subplots(2,1, figsize=(8, 8))

for num, alg in enumerate(paths):
    agents = []
    for i, (x, ys) in enumerate(paths[alg].items()):
        y_start = ys[0]
        y_end = ys[-1]
        y_min = min(ys)
        y_max = max(ys)

        # Draw the full range as a thin background line
        agents.append(x)
        ax[num].plot([i, i], [y_min, y_max], color="lightgray", linewidth=4, zorder=1)

        color = "gray"
        if str(x) in delays:
            color = "red"

        # Mark intermediate points
        for y in ys[1:-1]:
            ax[num].scatter(i, y, color=color, s=30, zorder=3)

        # Draw arrow from first to last value
        ax[num].annotate(
            "",
            xy=(i, y_end),
            xytext=(i, y_start),
            arrowprops=dict(arrowstyle="->", color="black", lw=2),
            zorder=2,
        )


    ax[num].set_xticks([int(x)-1 for x in agents])
    ax[num].set_xticklabels([str(x)  if i % 4 == 0 else "" for i,x in enumerate(agents)], fontsize=12)
    ax[num].set_xlabel("Agent", fontsize=12)
    ax[num].set_ylabel("Arrival Time", fontsize=12)
    ax[num].set_yticklabels(ax[num].get_yticklabels(), fontsize=12)
    ax[num].set_title(alg)
plt.tight_layout()
filepath = os.path.join(os.path.dirname(os.path.abspath("__file__")), "output", "sequential_delay_updates.png")
plt.savefig(filepath, dpi=600)
plt.show()